# Notebook 1: Data Curation
**Pipeline:** Scaffold-aware QSAR for *Candida albicans* antifungal activity
**Input:** Local ChEMBL 36 SQLite database
**Output:** `CAlbicans_curated.csv`, `curation_log.json`
**Reference:** Salihu & Wan Salleh, *J. Chem. Inf. Model.* 2026


In [ ]:
import sqlite3, os, json, warnings
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from scipy.stats import gmean
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── USER SETTINGS ─────────────────────────────────────────────────────────
# Download ChEMBL 36 SQLite from: https://chembl.gitbook.io/chembl-interface-documentation/downloads
# Set DB_PATH to the full path of your chembl_36.db file
DB_PATH  = 'path/to/chembl_36.db'   # <-- UPDATE THIS
OUT_DIR  = 'data'
os.makedirs(OUT_DIR, exist_ok=True)

# Curation constants (do not change unless replicating with different thresholds)
MIC_MIN, MIC_MAX = 0.001, 1024.0    # ug/mL; CLSI broth microdilution ceiling
ACTIVE_THRESH    = 8.0              # ug/mL CLSI susceptibility breakpoint
GREY_FACTOR      = 2.0              # 2-fold either side of threshold
MW_MIN, MW_MAX   = 100, 1000
print('Settings loaded. Ensure DB_PATH points to your local ChEMBL 36 SQLite file.')

In [ ]:
# Raw MIC extraction from ChEMBL 36
q = """
SELECT md.chembl_id AS compound_id, cs.canonical_smiles AS smiles,
       act.standard_value AS MIC_ugml, act.standard_type,
       act.pchembl_value AS pchembl, a.assay_id, a.description AS assay_desc
FROM target_dictionary td
JOIN assays a        ON td.tid = a.tid
JOIN activities act  ON a.assay_id = act.assay_id
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN compound_structures cs ON md.molregno = cs.molregno
WHERE td.chembl_id = 'CHEMBL366'
  AND act.standard_type IN ('MIC','MIC80')
  AND act.standard_relation = '='
  AND act.standard_units = 'ug.mL-1'
  AND act.standard_value IS NOT NULL
  AND cs.canonical_smiles IS NOT NULL
"""
conn   = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql_query(q, conn)
conn.close()
print(f'Raw records: {len(df_raw)} | Unique compounds: {df_raw["compound_id"].nunique()}')

In [ ]:
# Curation pipeline
def standardize(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return None
        mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except: return None

def get_scaffold(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return 'no_scaffold'
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except: return 'no_scaffold'

def get_props(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return None
        return {'MW': Descriptors.MolWt(mol), 'LogP': Descriptors.MolLogP(mol),
                'HBD': rdMolDescriptors.CalcNumHBD(mol),
                'HBA': rdMolDescriptors.CalcNumHBA(mol),
                'TPSA': Descriptors.TPSA(mol),
                'RotB': rdMolDescriptors.CalcNumRotatableBonds(mol),
                'HeavyAtoms': mol.GetNumHeavyAtoms()}
    except: return None

# Step 1: value range
df = df_raw[(df_raw['MIC_ugml'] >= MIC_MIN) & (df_raw['MIC_ugml'] <= MIC_MAX)].copy()
# Step 2: SMILES standardisation
df['std_smiles'] = df['smiles'].apply(standardize)
df = df[df['std_smiles'].notna()].copy()
# Step 3: geometric mean aggregation
df_agg = df.groupby('compound_id').agg(
    std_smiles=('std_smiles','first'),
    MIC_ugml=('MIC_ugml', lambda x: float(gmean(x))),
    n_records=('MIC_ugml','count')
).reset_index()
# Step 4: physicochemical filter
props = df_agg['std_smiles'].apply(get_props)
df_props = pd.DataFrame(props.tolist(), index=df_agg.index)
df_agg = pd.concat([df_agg, df_props], axis=1)
df_agg = df_agg[df_agg['MW'].notna() & (df_agg['MW']>=MW_MIN) & (df_agg['MW']<=MW_MAX) & (df_agg['HeavyAtoms']>=7)].copy()
# Step 5: binary labels + grey zone removal
grey_low, grey_high = ACTIVE_THRESH/GREY_FACTOR, ACTIVE_THRESH*GREY_FACTOR
df_agg['pMIC'] = -np.log10(df_agg['MIC_ugml']*1e-6)
df_binary = df_agg[~((df_agg['MIC_ugml']>grey_low)&(df_agg['MIC_ugml']<grey_high))].copy()
df_binary['label'] = (df_binary['MIC_ugml']<=ACTIVE_THRESH).astype(int)
# Step 6: Murcko scaffolds
df_binary['scaffold'] = df_binary['std_smiles'].apply(get_scaffold)
df_final = df_binary[['compound_id','std_smiles','scaffold','MIC_ugml','pMIC','label',
                       'MW','LogP','HBD','HBA','TPSA','RotB','n_records']].reset_index(drop=True)
print(f'Final dataset: {len(df_final)} compounds')
print(f'Active: {df_final["label"].sum()} ({df_final["label"].mean()*100:.1f}%) | '
      f'Inactive: {(df_final["label"]==0).sum()} ({(df_final["label"]==0).mean()*100:.1f}%)')
print(f'Unique scaffolds: {df_final["scaffold"].nunique()}')

In [ ]:
# Save outputs
out_csv = os.path.join(OUT_DIR, 'CAlbicans_curated.csv')
df_final.to_csv(out_csv, index=False)
log = {'raw_records': len(df_raw), 'raw_compounds': df_raw['compound_id'].nunique(),
       'final_compounds': len(df_final), 'final_active': int(df_final['label'].sum()),
       'final_inactive': int((df_final['label']==0).sum()),
       'unique_scaffolds': int(df_final['scaffold'].nunique()),
       'active_threshold_ugml': ACTIVE_THRESH, 'grey_zone_ugml': f'{grey_low}-{grey_high}'}
with open(os.path.join(OUT_DIR,'curation_log.json'),'w') as f:
    json.dump(log, f, indent=2)
print(f'Saved: {out_csv}')
print(json.dumps(log, indent=2))